# Evaluating a Classifier: Predicting Customer Churn

Last month, a telecom operator lost more than a quarter of its customers. It would like to spot the customers likely to leave (*churn*) ahead of time, to make them a retention offer. A model that predicts churn is only useful if we know how to evaluate it: that is what this lab is about.

The lab has two parts (about 1 h 15 for both):

1. **Evaluating a classifier** (exercises 1 to 4): a naive baseline, a logistic regression, its confusion matrix, its precision and its recall, then cross-validation. An introductory course can stop there.
2. **Comparing two models and choosing a threshold** (exercises 5 to 7): a naive Bayes classifier against the logistic regression, their ROC curves and their AUC, then the decision threshold chosen from the cost of the errors.

The data preparation is given ready-made: the exercises are about evaluation.

Data: [Telco Customer Churn](https://github.com/IBM/telco-customer-churn-on-icp4d) (IBM, Apache 2.0 license), 7,043 fictional customers.

## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
  ConfusionMatrixDisplay,
  RocCurveDisplay,
  accuracy_score,
  classification_report,
  precision_score,
  recall_score,
  roc_auc_score,
)
from sklearn.model_selection import (
  StratifiedKFold,
  cross_val_predict,
  cross_validate,
  train_test_split,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## The Data

Each row is a customer: their contract, their services, their bills, and the target, `Churned`, which is 1 if they left last month and 0 otherwise.

In [ ]:
customers = pd.read_csv(
  "https://raw.githubusercontent.com/shuuchuu/datasets/main/telco-churn/telco-churn.csv"
)

# Readable column names and values
columns = {
  "gender": "Gender",
  "SeniorCitizen": "Senior",
  "Partner": "Partner",
  "Dependents": "Dependents",
  "tenure": "Tenure (months)",
  "PhoneService": "Phone",
  "MultipleLines": "Multiple lines",
  "InternetService": "Internet",
  "OnlineSecurity": "Online security",
  "OnlineBackup": "Online backup",
  "DeviceProtection": "Device protection",
  "TechSupport": "Tech support",
  "StreamingTV": "TV streaming",
  "StreamingMovies": "Movie streaming",
  "Contract": "Contract",
  "PaperlessBilling": "Paperless billing",
  "PaymentMethod": "Payment method",
  "MonthlyCharges": "Monthly bill ($)",
  "TotalCharges": "Total billed ($)",
  "Churn": "Churned",
}
values = {
  "No phone service": "No phone",
  "No internet service": "No internet",
  "Fiber optic": "Fiber",
  "Month-to-month": "Month-to-month",
  "One year": "1 year",
  "Two year": "2 years",
  "Bank transfer (automatic)": "Automatic bank transfer",
  "Credit card (automatic)": "Automatic credit card",
}

customers = customers.drop(columns="customerID").rename(columns=columns).replace(values)
customers["Senior"] = customers["Senior"].map({0: "No", 1: "Yes"})
# New customers haven't paid anything yet: their total is blank
customers["Total billed ($)"] = pd.to_numeric(
  customers["Total billed ($)"], errors="coerce"
).fillna(0)
customers["Churned"] = (customers["Churned"] == "Yes").astype(int)

print(f"{len(customers)} customers, {customers['Churned'].mean():.1%} of whom churned")
customers.head()

We set a quarter of the customers aside for the **test**, the final measure: no choice will be made on them. The split is stratified (`stratify=y`): the churn rate is the same in the training set and in the test set.

In [ ]:
X = customers.drop(columns="Churned")
y = customers["Churned"]
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.25, stratify=y, random_state=0
)
print(f"Training: {len(X_train)} customers, test: {len(X_test)} customers")
print(f"Churn rate: {y_train.mean():.1%} (training), {y_test.mean():.1%} (test)")

The preprocessing, given ready-made: the categorical columns are one-hot encoded (one column of 0s and 1s per value; `handle_unknown="ignore"` ignores a value the training set hasn't seen), the three numeric columns are standardized. Put in a pipeline with the model, it is only fitted on the training data: nothing leaks from the test set into the model.

In [ ]:
numeric_columns = ["Tenure (months)", "Monthly bill ($)", "Total billed ($)"]
categorical_columns = [column for column in X.columns if column not in numeric_columns]

preprocessing = ColumnTransformer(
  [
    (
      "categorical",
      OneHotEncoder(handle_unknown="ignore", sparse_output=False),
      categorical_columns,
    ),
    ("numeric", StandardScaler(), numeric_columns),
  ]
)

n_columns = preprocessing.fit_transform(X_train).shape[1]
print(
  f"{len(categorical_columns)} categorical columns, "
  f"{len(numeric_columns)} numeric ones: {n_columns} after preprocessing"
)

## Part 1 — Evaluating a Classifier

One question guides this part: does the model spot the customers who are going to leave?

### 1. A Naive Baseline

Before any model, a baseline: a classifier that ignores the data and always predicts the most frequent class, here "stays". A model is only worth something if it beats it.

*Create a baseline `baseline` with `DummyClassifier`, which always predicts the majority class, train it on the training set, then print its accuracy on the test set.*

**Useful documentation**

- [`sklearn.dummy.DummyClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html)
- [`sklearn.metrics.accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)

In [ ]:
# Your code here

#### Additional hints

- `DummyClassifier(strategy="most_frequent")`: always predicts the training set's most frequent class.
- `.fit(X_train, y_train)`, then `.predict(X_test)`: the predictions, to pass to `accuracy_score(y_test, predictions)`, which returns a proportion between 0 and 1.
- Expected accuracy: the share of the test set's customers who stayed, 73.5%.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
baseline = DummyClassifier(strategy=...)
baseline.fit(..., ...)
print(f"Baseline accuracy: {accuracy_score(y_test, ...):.1%}")
```

#### Solution

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
print(f"Baseline accuracy: {accuracy_score(y_test, baseline.predict(X_test)):.1%}")

73.5% of right answers without learning anything: that is the share of customers who stayed. An 80% score looks good in itself; against this baseline, much less so. And the baseline spots no churner at all: exercise 3 measures it.

### 2. A Logistic Regression

*Create a model `model` that chains the preprocessing `preprocessing` and a `LogisticRegression` with `make_pipeline`, train it on the training set, then print its accuracy on the test set. How many points does it gain over the baseline?*

**Useful documentation**

- [`sklearn.linear_model.LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

In [ ]:
# Your code here

#### Additional hints

- `make_pipeline(preprocessing, LogisticRegression(max_iter=1000))`: `max_iter=1000` gives the optimization enough iterations to converge (the default, 100, may not be enough and raise a warning).
- The pipeline is used like a model: `model.fit(X_train, y_train)`, then `model.predict(X_test)`.
- Expected accuracy: about 79%.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
model = make_pipeline(preprocessing, LogisticRegression(max_iter=...))
model.fit(..., ...)
print(f"Logistic regression accuracy: {accuracy_score(y_test, ...):.1%}")
```

#### Solution

In [ ]:
model = make_pipeline(preprocessing, LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
print(
  f"Logistic regression accuracy: {accuracy_score(y_test, model.predict(X_test)):.1%}"
)

79.2% against 73.5%: less than 6 points better than the baseline. Is it a good model? Accuracy doesn't say where its errors are.

### 3. Confusion Matrix, Precision and Recall

*Plot the confusion matrix of `model` on the test set with `ConfusionMatrixDisplay.from_estimator`, then print the precision, the recall and the F1 score of each class with `classification_report`. How many of the test set's churners does the model miss? Finally, compute the baseline's recall with `recall_score`.*

In the matrix, as in the course, the rows are the truth and the columns the prediction; the positive class is 1, "churned".

**Useful documentation**

- [`sklearn.metrics.ConfusionMatrixDisplay.from_estimator`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_estimator)
- [`sklearn.metrics.classification_report`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)
- [`sklearn.metrics.recall_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)

In [ ]:
# Your code here

#### Additional hints

- `ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, display_labels=["Stayed", "Churned"])`: `display_labels` names the classes 0 and 1; `plt.show()` displays the figure.
- `classification_report(y_test, model.predict(X_test), target_names=["Stayed", "Churned"])` returns a text, to pass to `print`. Its `Churned` row gives the precision, the recall and the F1 of the churners.
- The churners the model misses are the false negatives: the cell in the "Churned" row and the "Stayed" column.
- `recall_score(y_test, baseline.predict(X_test))`: the recall of class 1.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
ConfusionMatrixDisplay.from_estimator(
  model, ..., ..., display_labels=["Stayed", "Churned"]
)
plt.show()
print(classification_report(y_test, ..., target_names=["Stayed", "Churned"]))
print(f"Baseline recall: {recall_score(y_test, ...):.1%}")
```

#### Solution

In [ ]:
ConfusionMatrixDisplay.from_estimator(
  model, X_test, y_test, display_labels=["Stayed", "Churned"]
)
plt.show()
print(
  classification_report(
    y_test, model.predict(X_test), target_names=["Stayed", "Churned"]
  )
)
print(f"Baseline recall: {recall_score(y_test, baseline.predict(X_test)):.1%}")

Of the test set's 467 churners, the model spots 248 (a recall of 53%) and misses 219: nearly one in two. When it predicts churn, it is right 63 times out of 100 (a precision of 63%). The baseline spots no churner at all: its recall is zero, despite its 73.5% accuracy. On imbalanced classes, accuracy is misleading: look at the precision and the recall of the class that matters.

### 4. Cross-Validation

A single split between training and test gives a score that depends on which customers were drawn into the test set. Cross-validation splits the training set into 5 folds, trains 5 models and gives the mean and the standard deviation of the 5 validation scores: enough to compare models without touching the test set, kept for the final measure.

*Create a split `folds` into 5 stratified, shuffled folds with `StratifiedKFold`, then write a function `evaluate(classifier, scores)` that evaluates `classifier` by cross-validation on the training set with `cross_validate`, for each score of the list `scores` (scikit-learn score names, such as `"recall"`), and returns a `DataFrame` with one row per score, holding the mean and the standard deviation over the 5 folds. The next section compares the baseline and the logistic regression.*

**Useful documentation**

- [`sklearn.model_selection.StratifiedKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)
- [`sklearn.model_selection.cross_validate`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html)
- [scikit-learn's score names](https://scikit-learn.org/stable/modules/model_evaluation.html#string-name-scorers)

In [ ]:
def evaluate(classifier, scores):
  # Your code here
  ...

#### Additional hints

- `StratifiedKFold(n_splits=5, shuffle=True, random_state=0)`: 5 folds that each keep the churn rate, drawn after a reproducible shuffle. Create `folds` before the function: exercise 7 uses it too.
- `cross_validate(classifier, X_train, y_train, cv=folds, scoring=scores)` returns a dictionary: for each score `s`, the key `f"test_{s}"` gives an array of 5 values, one per fold.
- `.mean()` and `.std()` of a NumPy array: its mean and its standard deviation.
- `pd.DataFrame({"mean": [...], "std": [...]}, index=scores)`: one row per score.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
folds = StratifiedKFold(n_splits=5, shuffle=..., random_state=0)


def evaluate(classifier, scores):
  results = cross_validate(classifier, X_train, y_train, cv=..., scoring=...)
  return pd.DataFrame(
    {
      "mean": [results[f"test_{score}"].mean() for score in scores],
      "std": [... for score in scores],
    },
    index=scores,
  )
```

#### Solution

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


def evaluate(classifier, scores):
  results = cross_validate(classifier, X_train, y_train, cv=folds, scoring=scores)
  return pd.DataFrame(
    {
      "mean": [results[f"test_{score}"].mean() for score in scores],
      "std": [results[f"test_{score}"].std() for score in scores],
    },
    index=scores,
  )

### The Baseline against the Logistic Regression

`evaluate` compares the two models of part 1, on accuracy, precision, recall and F1:

In [ ]:
part_1_scores = ["accuracy", "precision", "recall", "f1"]
pd.concat(
  {
    "Baseline": evaluate(baseline, part_1_scores),
    "Logistic regression": evaluate(model, part_1_scores),
  },
  axis=1,
).round(3)

In cross-validation, the logistic regression gets 80.7% accuracy (± 2.2 points from one fold to the next), a precision of 66% (± 5) and a recall of 55% (± 3); the baseline, 73.5% accuracy and zeros everywhere else (scikit-learn warns that its precision is undefined, since it never predicts churn, and sets it to 0). The standard deviation tells what a difference is worth: a gap between two models smaller than these fold-to-fold variations isn't enough to tell them apart.

These scores aren't the test set's (79.2% accuracy, 53% recall): they are other customers, and the gap stays within the fold-to-fold margin.

## Part 2 — Comparing Two Models and Choosing a Threshold

The logistic regression isn't the only possible classifier. How do we compare two of them? And how do we go from a churn probability to a decision: make this customer an offer, or not?

If your course stops at part 1, go to the conclusion.

### 5. A Second Classifier: Naive Bayes

*Create a model `bayes` that chains the same preprocessing and a `GaussianNB` (the naive Bayes classifier for numeric variables), then compare it to `model` with `evaluate`, on the scores `"roc_auc"` and `"recall"`. Which one spots the most churners? Which one has the best AUC?*

The AUC (`roc_auc`) is the area under the ROC curve, which exercise 6 plots: it measures whether the model gives a higher churn probability to the customers who left than to the others, whatever the threshold.

**Useful documentation**

- [`sklearn.naive_bayes.GaussianNB`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html)

In [ ]:
# Your code here

#### Additional hints

- `make_pipeline(preprocessing, GaussianNB())`: like `model`, with another classifier.
- `evaluate` trains one model per fold itself: no need to call `bayes.fit` here.
- `pd.concat({"Logistic regression": ..., "Naive Bayes": ...}, axis=1)` puts the two tables side by side, like the cell after exercise 4.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
bayes = make_pipeline(preprocessing, ...)
part_2_scores = ["roc_auc", "recall"]
pd.concat(
  {
    "Logistic regression": evaluate(model, part_2_scores),
    "Naive Bayes": evaluate(..., ...),
  },
  axis=1,
).round(3)
```

#### Solution

In [ ]:
bayes = make_pipeline(preprocessing, GaussianNB())
part_2_scores = ["roc_auc", "recall"]
pd.concat(
  {
    "Logistic regression": evaluate(model, part_2_scores),
    "Naive Bayes": evaluate(bayes, part_2_scores),
  },
  axis=1,
).round(3)

Naive Bayes spots many more churners (a recall of 84% against 55%), but its AUC is lower (0.820 against 0.845): it ranks the customers, from the riskiest to the safest, less well. Its recall comes from predicting churn often, wrongly more than half the time (a precision of 46%): a recall only reads together with the precision, and both depend on the 0.5 threshold that `predict` applies. The AUC depends on no threshold.

Any other classifier seen in the course (k-nearest neighbors, random forest…) goes in place of `GaussianNB()` in `make_pipeline`, with nothing else to change: the preprocessing and `evaluate` stay the same.

### 6. ROC Curves

*Train `bayes` on the training set, then plot on the same axes the ROC curves of both models on the test set with `RocCurveDisplay.from_estimator`, and print the AUC of each with `roc_auc_score`. Which one should we keep? Why doesn't the AUC depend on the threshold?*

**Useful documentation**

- [`sklearn.metrics.RocCurveDisplay.from_estimator`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.RocCurveDisplay.html#sklearn.metrics.RocCurveDisplay.from_estimator)
- [`sklearn.metrics.roc_auc_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)

In [ ]:
# Your code here

#### Additional hints

- `fig, ax = plt.subplots(figsize=(6, 6))`, then `RocCurveDisplay.from_estimator(model, X_test, y_test, name="Logistic regression", ax=ax)`: `ax=ax` plots on these axes, `name` labels the curve in the legend. Same call for `bayes`; `plot_chance_level=True` on the second adds the chance diagonal.
- `roc_auc_score(y_test, probas)` takes the churn probabilities, not the predicted classes: `classifier.predict_proba(X_test)[:, 1]`, the second column being the probability of class 1.
- Expected AUCs: about 0.84 for the logistic regression and 0.81 for naive Bayes.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
bayes.fit(..., ...)

fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_estimator(model, X_test, y_test, name="Logistic regression", ax=ax)
RocCurveDisplay.from_estimator(
  bayes, ..., ..., name="Naive Bayes", ax=..., plot_chance_level=True
)
plt.show()

for name, classifier in [("Logistic regression", model), ("Naive Bayes", bayes)]:
  auc = roc_auc_score(y_test, classifier.predict_proba(X_test)[...])
  print(f"AUC, {name}: {auc:.3f}")
```

#### Solution

In [ ]:
bayes.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_estimator(model, X_test, y_test, name="Logistic regression", ax=ax)
RocCurveDisplay.from_estimator(
  bayes, X_test, y_test, name="Naive Bayes", ax=ax, plot_chance_level=True
)
plt.show()

for name, classifier in [("Logistic regression", model), ("Naive Bayes", bayes)]:
  auc = roc_auc_score(y_test, classifier.predict_proba(X_test)[:, 1])
  print(f"AUC, {name}: {auc:.3f}")

The logistic regression's curve runs above the naive Bayes one over nearly its whole length, and its AUC is higher on the test set too (0.844 against 0.809): we keep the logistic regression.

Each point of a curve is a threshold: lowering it spots more churners (moving up) at the price of more false alarms (moving right). The AUC sums up the whole curve, so every threshold at once: it measures how the model orders the customers (it is the probability that a churner drawn at random gets a higher churn probability than a customer who stayed, drawn at random), not the decision we draw from it. Choosing that threshold is what's left.

### 7. The Threshold from the Cost of Errors

The 0.5 threshold is nothing special: the cost of the errors sets it. Let's assume, for the operator (these figures are assumptions, not data):

- a customer who leaves without having received an offer loses $500 of revenue (about 7 months of a churner's bill);
- a retention offer costs $100 for each customer who receives it, whether they would have left or not, and surely keeps a customer who was about to leave.

The total cost of a threshold is therefore $500 per churner without an offer (a false negative) and $100 per offer (a true or a false positive).

The threshold is chosen on the training set, without touching the test set. That takes probabilities predicted for customers the model hasn't seen: `cross_val_predict` gives them, each customer getting the probability predicted by the model of the fold where it was used for validation.

*Write the function `total_cost(y_true, proba, threshold)`, which returns the total cost of making an offer to each customer whose churn probability `proba` reaches `threshold`. Then compute the out-of-sample probabilities `train_proba` of `model` on the training set with `cross_val_predict` and `folds`, plot the total cost for the thresholds from 0.05 to 0.95 in steps of 0.05, and keep in `best_threshold` the threshold that minimizes it. The next section applies it to the test set.*

**Useful documentation**

- [`sklearn.model_selection.cross_val_predict`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html)

In [ ]:
CHURN_COST = 500  # $: a customer who left without having received an offer
OFFER_COST = 100  # $: a retention offer


def total_cost(y_true, proba, threshold): ...


# Your code here

#### Additional hints

- In `total_cost`: `offer = proba >= threshold` gives an array of booleans, true for each customer who receives an offer. The missed churners are `(np.asarray(y_true) == 1) & ~offer` (`~`: the negation), and `.sum()` counts the trues of an array of booleans. A small example to check: `total_cost(np.array([1, 0, 1]), np.array([0.9, 0.6, 0.2]), 0.5)` is 700 (two offers and one missed churner).
- `cross_val_predict(model, X_train, y_train, cv=folds, method="predict_proba")` returns two columns, like `predict_proba`: `[:, 1]` keeps the churn probability.
- `np.linspace(0.05, 0.95, 19)`: the 19 thresholds from 0.05 to 0.95; `np.argmin(costs)`: the position of the lowest cost.
- Expected best threshold: 0.15, with a cost almost as low from 0.15 to 0.25.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def total_cost(y_true, proba, threshold):
  offer = proba >= threshold
  missed_churners = (np.asarray(y_true) == 1) & ~offer
  return CHURN_COST * ... + OFFER_COST * ...


train_proba = cross_val_predict(model, X_train, y_train, cv=folds, method=...)[:, 1]
thresholds = np.linspace(0.05, 0.95, 19)
costs = [total_cost(..., ..., threshold) for threshold in thresholds]
best_threshold = thresholds[...]

fig, ax = plt.subplots()
ax.plot(thresholds, costs, marker="o")
ax.axvline(best_threshold, color="tab:red", linestyle="--")
ax.set_xlabel("Threshold")
ax.set_ylabel("Total cost on the training set ($)")
plt.show()
print(f"Best threshold: {best_threshold:.2f}")
```

#### Solution

In [ ]:
def total_cost(y_true, proba, threshold):
  offer = proba >= threshold
  missed_churners = (np.asarray(y_true) == 1) & ~offer
  return CHURN_COST * missed_churners.sum() + OFFER_COST * offer.sum()


train_proba = cross_val_predict(
  model, X_train, y_train, cv=folds, method="predict_proba"
)[:, 1]
thresholds = np.linspace(0.05, 0.95, 19)
costs = [total_cost(y_train, train_proba, threshold) for threshold in thresholds]
best_threshold = thresholds[np.argmin(costs)]

fig, ax = plt.subplots()
ax.plot(thresholds, costs, marker="o")
ax.axvline(best_threshold, color="tab:red", linestyle="--")
ax.set_xlabel("Threshold")
ax.set_ylabel("Total cost on the training set ($)")
plt.show()
print(f"Best threshold: {best_threshold:.2f}")

### The Chosen Threshold, on the Test Set

`best_threshold`, applied to the test set, against the 0.5 threshold and two landmarks: making no offer at all, or making one to every customer.

In [ ]:
test_proba = model.predict_proba(X_test)[:, 1]
print(f"No offer at all: ${CHURN_COST * y_test.sum()}")
print(f"An offer to every customer: ${OFFER_COST * len(y_test)}")
pd.DataFrame(
  [
    {
      "Threshold": threshold,
      "Offers": (test_proba >= threshold).sum(),
      "Total cost ($)": total_cost(y_test, test_proba, threshold),
      "Precision": precision_score(y_test, test_proba >= threshold),
      "Recall": recall_score(y_test, test_proba >= threshold),
    }
    for threshold in [0.5, best_threshold]
  ]
).round(2)

On the test set, the 0.15 threshold costs $114,100, against $149,100 at the 0.5 threshold: $35,000 saved, and much less than making no offer at all ($233,500) or making one to everyone ($176,100). The model then makes 956 offers instead of 396: its precision drops from 63% to 45%, but its recall rises from 53% to 92%. An offer costs five times less than a missed churner: better many false alarms than one more customer lost.

This threshold depends on the assumptions: with other costs, redo the computation. In fact, an offer pays off as soon as the churn probability exceeds 100 / 500 = 0.2: if the model's probabilities are reliable, the best threshold is close to it, and the curve is nearly flat around it. scikit-learn (1.5 and later) makes this choice in one class, [`TunedThresholdClassifierCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TunedThresholdClassifierCV.html), which tunes the threshold by cross-validation from a given score, a business cost included.

## Conclusion

- On imbalanced classes, accuracy is misleading: a baseline that spots no churner gets 73.5%. Look at the confusion matrix, and at the precision and the recall of the class that matters.
- To compare models, cross-validation on the training set: a mean, and a standard deviation that tells what a difference is worth. The test set stays for the final measure.
- The AUC ranks the models without depending on the threshold: it measures how they order the customers (part 2).
- The threshold is decided from the business cost of the errors, on out-of-sample probabilities of the training set: here 0.15 rather than 0.5 (part 2).